# 05 - Kaggle M3.1 detector recovery

End-to-end Kaggle runner for M3.1 detector recovery. This notebook is research-only. It does not modify the frozen classifier bundle or overwrite the existing TXL-PBC detector.

The notebook assumes you attach/upload the required datasets and model artifacts to Kaggle first. It intentionally does not invent or auto-accept LeukemiaAttri download URLs, licenses, file sizes, or checksums.

## 0. Required Kaggle inputs

Attach Kaggle datasets or upload files so these paths can be resolved:

- Repository checkout containing `src/`, `configs/`, `scripts/`, `tests/`, and notebooks.
- Existing detector bundle or at least `models/txl-pbc-yolo26n-v0.1/weights.pt`.
- TXL-PBC raw dataset or existing WBC-only YOLO derivative.
- LeukemiaAttri extracted dataset after you have reviewed its terms/license.
- Optional reviewed target labels for `wbc.jpg` and `wbc2.jpeg`.

If LeukemiaAttri is not legally available in the runner, stop before training. The code can still run setup, local tests, target failure evidence, and dry-run training plans.

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

def run(command: str) -> None:
    print(f'\n$ {command}')
    subprocess.run(command, shell=True, check=True)

def first_existing(paths: list[str]) -> Path | None:
    for raw in paths:
        path = Path(raw)
        if path.exists():
            return path
    return None

print('Kaggle input roots:')
for path in sorted(Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []:
    print(' -', path)

## 1. Locate or copy the repo

Edit `REPO_INPUT` if your attached repo dataset has a different path. If you already opened this notebook inside the repo checkout, leave it as `None`.

In [ ]:
REPO_INPUT = first_existing([
    '/kaggle/input/blood',
    '/kaggle/input/blood-repo',
    '/kaggle/input/wbc-blood-film-analyzer',
])
WORKDIR = Path('/kaggle/working/blood')

if Path('src/bloodfilm').exists():
    WORKDIR = Path.cwd()
elif REPO_INPUT is not None:
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    shutil.copytree(REPO_INPUT, WORKDIR, ignore=shutil.ignore_patterns('.git', '.venv*', '__pycache__'))
else:
    raise RuntimeError('Attach the repo as a Kaggle dataset or run this notebook from the repo root.')

os.chdir(WORKDIR)
print('Working directory:', WORKDIR)
assert Path('src/bloodfilm').exists()

## 2. Install dependencies and verify GPU

In [ ]:
!pip install -q ultralytics Pillow numpy pandas matplotlib seaborn PyYAML pytest ruff mypy timm torchvision tqdm
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Restore required model artifacts

The existing TXL-PBC detector is used only as Experiment A initialization. It must not be overwritten.

In [ ]:
DETECTOR_WEIGHTS = first_existing([
    'models/txl-pbc-yolo26n-v0.1/weights.pt',
    '/kaggle/input/txl-pbc-yolo26n-v01/weights.pt',
    '/kaggle/input/txl-pbc-yolo26n-v0-1/weights.pt',
])
if DETECTOR_WEIGHTS is None:
    raise RuntimeError('Attach models/txl-pbc-yolo26n-v0.1/weights.pt as a Kaggle dataset.')
target = Path('models/txl-pbc-yolo26n-v0.1/weights.pt')
target.parent.mkdir(parents=True, exist_ok=True)
if DETECTOR_WEIGHTS.resolve() != target.resolve():
    shutil.copy2(DETECTOR_WEIGHTS, target)
print('Experiment A initial weights:', target)

## 4. Locate datasets

Edit the candidate path lists if your Kaggle dataset names differ. LeukemiaAttri must be attached after license/terms review. TXL-PBC may be attached as raw data or as the WBC-only YOLO derivative.

In [ ]:
TXL_PBC_RAW = first_existing([
    '/kaggle/input/txl-pbc/TXL-PBC',
    '/kaggle/input/txl-pbc-dataset/TXL-PBC',
    'data/raw/TXL-PBC/TXL-PBC',
])
TXL_PBC_WBC_YOLO = first_existing([
    '/kaggle/input/txl-pbc-wbc-yolo',
    'data/detection/txl-pbc-wbc-yolo',
])
LEUKEMIA_ATTRI = first_existing([
    '/kaggle/input/leukemia-attri',
    '/kaggle/input/blood-cancer-dataset-leukemia-attri-miccai-2024',
    'data/raw/LeukemiaAttri',
])
print('TXL_PBC_RAW:', TXL_PBC_RAW)
print('TXL_PBC_WBC_YOLO:', TXL_PBC_WBC_YOLO)
print('LEUKEMIA_ATTRI:', LEUKEMIA_ATTRI)
if LEUKEMIA_ATTRI is None:
    print('LeukemiaAttri not attached. You can continue through dry-run planning, but not real multidomain training.')

## 5. Run repo verification on Kaggle

This catches import/config breakage before starting long GPU jobs.

In [ ]:
!PYTHONPATH=src python -m pytest -q tests/unit/test_detector_training_plan.py tests/unit/test_detector_annotations.py tests/unit/test_detector_geometry.py tests/unit/test_detector_sweep.py
!PYTHONPATH=src python -m bloodfilm.cli config validate --config configs/detector_multidomain.yaml || true

## 6. Audit LeukemiaAttri once layout is known

The exact class names and annotation path must come from the attached dataset. Update `LEUKEMIA_IMAGE_ROOT`, `LEUKEMIA_LABEL_ROOT`, and class mappings after inspecting the dataset. Do not train until this audit has been reviewed.

In [ ]:
if LEUKEMIA_ATTRI is not None:
    for path in sorted(Path(LEUKEMIA_ATTRI).glob('*'))[:50]:
        print(path)
else:
    print('Attach LeukemiaAttri before running this audit section.')

In [ ]:
# Edit these after inspecting LeukemiaAttri layout.
LEUKEMIA_IMAGE_ROOT = Path(LEUKEMIA_ATTRI) / 'images' if LEUKEMIA_ATTRI else None
LEUKEMIA_LABEL_ROOT = Path(LEUKEMIA_ATTRI) / 'labels' if LEUKEMIA_ATTRI else None
LEUKEMIA_CLASS_ARGS = [
    '--class-names', '0=Neutrophil',
    '--class-mapping', 'Neutrophil=candidate_wbc',
]
if LEUKEMIA_IMAGE_ROOT and LEUKEMIA_IMAGE_ROOT.exists() and LEUKEMIA_LABEL_ROOT and LEUKEMIA_LABEL_ROOT.exists():
    command = [
        'PYTHONPATH=src python -m bloodfilm.cli detector audit',
        '--format leukemia-attri-yolo',
        f'--image-root {LEUKEMIA_IMAGE_ROOT}',
        f'--label-root {LEUKEMIA_LABEL_ROOT}',
        ' '.join(LEUKEMIA_CLASS_ARGS),
        '--report-output outputs/reports/leukemia_attri_audit.json',
    ]
    run(' '.join(command))
else:
    print('LeukemiaAttri image/label roots not found. Update paths above after inspecting attached data.')

## 7. Build or attach unified YOLO data.yaml

If a reviewed unified YOLO dataset is already attached as a Kaggle Dataset, point `MULTIDOMAIN_DATA_YAML` to it. Otherwise this section can build `data/detection/multidomain/data.yaml` from attached TXL-PBC and LeukemiaAttri YOLO roots. Keep `RUN_MULTIDOMAIN_PREPARE = False` until every class ID/name has been inspected against the real LeukemiaAttri source metadata.

In [ ]:
MULTIDOMAIN_DATA_YAML = first_existing([
    '/kaggle/input/blood-detector-multidomain-yolo/data.yaml',
    'data/detection/multidomain/data.yaml',
])
TXL_PBC_ROOT = first_existing([
    '/kaggle/input/txl-pbc/TXL-PBC',
    '/kaggle/input/txl-pbc',
    'data/raw/TXL-PBC/TXL-PBC',
    'data/raw/TXL-PBC',
])
LEUKEMIA_ATTRI_ROOT = first_existing([
    '/kaggle/input/leukemia-attri',
    'data/raw/LeukemiaAttri',
])
RUN_MULTIDOMAIN_PREPARE = False  # Set True only after reviewing all source class IDs and mappings.
if RUN_MULTIDOMAIN_PREPARE and MULTIDOMAIN_DATA_YAML is None and TXL_PBC_ROOT and LEUKEMIA_ATTRI_ROOT:
    prepare_parts = [
        'PYTHONPATH=src python -m bloodfilm.cli detector prepare',
        f'--txl-pbc-root {TXL_PBC_ROOT}',
        f'--leukemia-attri-root {LEUKEMIA_ATTRI_ROOT}',
        '--output-root data/detection/multidomain',
        '--manifest-output data/manifests/detector_multidomain_manifest.csv',
        '--splits-output data/manifests/detector_multidomain_splits.csv',
        '--leakage-output outputs/reports/detector_multidomain_leakage.json',
        '--report-output outputs/reports/detector_multidomain_preparation.json',
        '--txl-class-names 0=wbc',
        '--txl-class-mapping wbc=candidate_wbc',
        # Replace/add these after inspecting the attached LeukemiaAttri data.yaml/classes.
        '--leukemia-class-names 0=Neutrophil',
        '--leukemia-class-mapping Neutrophil=candidate_wbc',
    ]
    run(' '.join(prepare_parts))
    MULTIDOMAIN_DATA_YAML = Path('data/detection/multidomain/data.yaml')
if MULTIDOMAIN_DATA_YAML is None:
    print('No unified multidomain YOLO data.yaml found yet.')
    print('Attach a prepared multidomain Kaggle Dataset, or set RUN_MULTIDOMAIN_PREPARE=True after class mapping review.')
else:
    print('Using multidomain data.yaml:', MULTIDOMAIN_DATA_YAML)

## 8. Dry-run the two training experiments

This always runs and prints the exact Experiment A/B commands.

In [ ]:
data_yaml_for_plan = MULTIDOMAIN_DATA_YAML or Path('data/detection/multidomain/data.yaml')
!PYTHONPATH=src python scripts/kaggle_train_detector_multidomain.py --data-yaml {data_yaml_for_plan} --project-dir outputs/detector_multidomain --dry-run

## 9. Run training on GPU

Only run this cell when `MULTIDOMAIN_DATA_YAML` points to a real reviewed fully annotated dataset. It trains both Experiment A and Experiment B.

In [ ]:
RUN_TRAINING = False  # Set True only after audit + reviewed multidomain data.yaml exist.
if RUN_TRAINING:
    assert MULTIDOMAIN_DATA_YAML is not None, 'No multidomain data.yaml available.'
    assert torch.cuda.is_available(), 'Enable GPU accelerator first.'
    run(f'PYTHONPATH=src python scripts/kaggle_train_detector_multidomain.py --data-yaml {MULTIDOMAIN_DATA_YAML} --project-dir outputs/detector_multidomain')
else:
    print('Training disabled. Set RUN_TRAINING=True after data audit passes.')

## 10. Target failure evidence and ROI smoke checks

These commands can be run when `data/microscope_samples/wbc.jpg` and `wbc2.jpeg` are present. They do not prove acceptance; they preserve failure evidence and preprocessing behavior.

In [ ]:
if Path('data/microscope_samples/wbc2.jpeg').exists():
    run('PYTHONPATH=src python -m bloodfilm.cli detector preprocess --input data/microscope_samples/wbc2.jpeg --report-output outputs/reports/detector_preprocess_wbc2.json')
    run('PYTHONPATH=src python -m bloodfilm.cli detector sweep --weights models/txl-pbc-yolo26n-v0.1/weights.pt --input data/microscope_samples/wbc2.jpeg --thresholds "0.05,0.10,0.15,0.25,0.35,0.50" --mode field_roi --report-output outputs/reports/detector_sweep_microscope_wbc2_field_roi.json')
else:
    print('Target microscope samples are not attached in this Kaggle session.')

## 11. Download these outputs back to the repo

After a real training run, download/copy back:

- `outputs/detector_multidomain/`
- `outputs/reports/detector_multidomain_kaggle_plan.json`
- `outputs/reports/leukemia_attri_audit.json`
- any generated evaluation/target-smoke reports

Do not package `models/wbc-detector-multidomain-yolo26n-v0.2/` until evaluation reports truthfully establish which experiment won and the target smoke tests pass.

In [ ]:
print('Artifacts currently available:')
for path in [Path('outputs/detector_multidomain'), Path('outputs/reports')]:
    if path.exists():
        print(path)
        for child in sorted(path.glob('*'))[:50]:
            print(' -', child)